# Stage 2.2 — Frozen visual memory and exact retrieval
Run in order on a fresh Colab Enterprise Linux Python 3.10–3.12 runtime, preferably V100 16 GB. **Cloud execution is UNVERIFIED until you run this notebook.** SmolVLA is never loaded or modified. The memory uses only the original 40 training trajectories; retrieval diagnostics use all 10 held-out trajectories.

Publish the Stage 2.2 commit or upload/unzip this checkout before starting. Git does **not** contain the excluded demonstration HDF5 files or memory NPZ. Configure a local uploaded dataset directory, a Stage 2.1 archive (local or GCS), or let the notebook reproduce Stage 2.1 from its pinned public source. Existing data is reused, never overwritten.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import sys, os, platform, subprocess, json, shutil
REPO_DIR = Path.home() / "cs6758-vla-project"
SOURCE_REF = "stage2-memory"  # Prefer your published Stage 2.2 commit SHA.
REPO_URL = "https://github.com/wcysunrise/cs6758-vla-project.git"
LOCAL_DATA_DIR = ""  # Uploaded directory containing both HDF5 files and both Stage 2.1 JSON files.
LOCAL_STAGE2_1_ZIP = ""  # Or uploaded Stage 2.1 notebook export ZIP.
GCS_STAGE2_1_ZIP = ""  # Or gs://bucket/path/stage2_1_export.zip.
GCS_OUTPUT_URI = ""  # Prefer gs://YOUR_BUCKET/cs6758/stage2_2; uses runtime ADC.
OUTPUT_ROOT = Path.home() / "stage2_2_outputs"
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
RUN_DIR = OUTPUT_ROOT / RUN_ID
DEVICE = "cuda"  # Explicitly choose "cpu" if no GPU is attached.
BATCH_SIZE = 32
VISUAL_WEIGHT, STATE_WEIGHT = 1.0, 1.0
if not REPO_DIR.exists():
    subprocess.run(["git","clone",REPO_URL,str(REPO_DIR)],check=True)
    subprocess.run(["git","checkout",SOURCE_REF],cwd=REPO_DIR,check=True)
assert (REPO_DIR/"memory/build_memory.py").exists(), "Publish Stage 2.2 or upload its checkout first"
RUN_DIR.mkdir(parents=True,exist_ok=False)
print("Outputs:",RUN_DIR)
if (REPO_DIR/".git").exists():
    (RUN_DIR/"git_commit.txt").write_text(subprocess.check_output(["git","rev-parse","HEAD"],cwd=REPO_DIR,text=True))

In [ ]:
assert platform.system()=="Linux"
assert (3,10)<=sys.version_info[:2]<=(3,12), "Use Python 3.10–3.12"
print(sys.version,platform.platform())
if shutil.which("nvidia-smi"): subprocess.run(["nvidia-smi"],check=True)
VENV=REPO_DIR/".venv-stage2_2"
subprocess.run([sys.executable,"-m","venv",str(VENV)],check=True)
PYTHON=str(VENV/"bin/python")
subprocess.run([PYTHON,"-m","pip","install","torch==2.7.1","torchvision==0.22.1","--index-url","https://download.pytorch.org/whl/cu126"],check=True)
subprocess.run([PYTHON,"-m","pip","install","-r",str(REPO_DIR/"requirements-stage2_2.txt")],check=True)
if GCS_OUTPUT_URI or GCS_STAGE2_1_ZIP:
    subprocess.run([PYTHON,"-m","pip","install","google-cloud-storage==3.4.0"],check=True)
subprocess.run([PYTHON,"-m","pip","check"],check=True)
(RUN_DIR/"installed_versions.txt").write_text(subprocess.check_output([PYTHON,"-m","pip","freeze"],text=True))

In [ ]:
preflight=r"""
import torch,torchvision,json,sys,platform
from pathlib import Path
info={'python':sys.version,'platform':platform.platform(),'torch':torch.__version__,'torchvision':torchvision.__version__,
      'cuda_build':torch.version.cuda,'cuda_available':torch.cuda.is_available(),'architectures':torch.cuda.get_arch_list()}
assert torch.__version__.split('+')[0]=='2.7.1'
assert torchvision.__version__.split('+')[0]=='0.22.1'
if torch.cuda.is_available():
    info['gpu']=torch.cuda.get_device_name(0);info['capability']=list(torch.cuda.get_device_capability(0))
    if info['capability']==[7,0]: assert 'sm_70' in info['architectures'], 'Wheel lacks V100 native support'
    x=torch.ones((64,64),device='cuda');y=x@x;torch.cuda.synchronize()
    assert torch.all(y==64).item();info['cuda_matmul']='PASS'
else: info['cuda_matmul']='UNVERIFIED: no CUDA'
Path(sys.argv[1]).write_text(json.dumps(info,indent=2));print(json.dumps(info,indent=2))
if sys.argv[2]=='cuda': assert torch.cuda.is_available(), 'Attach GPU or explicitly choose DEVICE=cpu'
"""
subprocess.run([PYTHON,"-c",preflight,str(RUN_DIR/"hardware.json"),DEVICE],check=True)
def run_logged(arguments, filename):
    with (RUN_DIR/filename).open("w") as log:
        result=subprocess.run([PYTHON,*arguments],cwd=REPO_DIR,stdout=log,stderr=subprocess.STDOUT)
    print((RUN_DIR/filename).read_text()[-12000:])
    result.check_returncode()

In [ ]:
# Access/upload fallback: point LOCAL_DATA_DIR or LOCAL_STAGE2_1_ZIP at files
# uploaded through the Colab Enterprise file browser. No Git-data assumption.
archive=Path(LOCAL_STAGE2_1_ZIP) if LOCAL_STAGE2_1_ZIP else None
if GCS_STAGE2_1_ZIP:
    archive=RUN_DIR/"stage2_1_input.zip"
    download=r"""
from google.cloud import storage
from urllib.parse import urlparse
import sys
u=urlparse(sys.argv[1]);assert u.scheme=='gs' and u.netloc
storage.Client().bucket(u.netloc).blob(u.path.lstrip('/')).download_to_filename(sys.argv[2],checksum='auto')
"""
    try:
        subprocess.run([PYTHON,"-c",download,GCS_STAGE2_1_ZIP,str(archive)],check=True)
    except subprocess.CalledProcessError:
        archive=None
        print("GCS input unavailable: will use local data if configured, otherwise reproduce pinned Stage 2.1.")
if archive:
    import zipfile
    destination=RUN_DIR/"stage2_1_import"
    destination.mkdir()
    with zipfile.ZipFile(archive) as z:
        for member in z.infolist():
            resolved=(destination/member.filename).resolve()
            assert resolved.is_relative_to(destination.resolve()), "Unsafe ZIP path"
        z.extractall(destination)
    candidates=list(destination.rglob("split_manifest.json"))
    assert len(candidates)==1, "Expected exactly one Stage 2.1 dataset in the archive"
    DATA_DIR=candidates[0].parent
elif LOCAL_DATA_DIR:
    DATA_DIR=Path(LOCAL_DATA_DIR).expanduser().resolve()
    assert DATA_DIR.is_dir(), "Upload Stage 2.1 data and correct LOCAL_DATA_DIR"
elif (REPO_DIR/"data/stage2_1_verified/split_manifest.json").exists():
    DATA_DIR=REPO_DIR/"data/stage2_1_verified"
else:
    DATA_DIR=RUN_DIR/"stage2_1_data"
    run_logged(["-m","memory.extract_demonstrations","--download","--raw-dir",str(OUTPUT_ROOT/"raw"),"--output",str(DATA_DIR)],"extraction.log")
run_logged(["-c","from memory.build_memory import validate_split; import sys; validate_split(sys.argv[1]); print('Verified original 40/10 split')",str(DATA_DIR)],"data_validation.log")
print("Using:",DATA_DIR)

In [ ]:
# This uses the genuine frozen ImageNet weights. First run downloads ~45 MB.
MEMORY_DIR=RUN_DIR/"memory"
run_logged(["-m","memory.build_memory","--data-dir",str(DATA_DIR),"--output-dir",str(MEMORY_DIR),"--device",DEVICE,"--batch-size",str(BATCH_SIZE)],"build.log")
run_logged(["-m","memory.retrieve","--data-dir",str(DATA_DIR),"--memory-dir",str(MEMORY_DIR),"--output",str(MEMORY_DIR/"heldout_retrieval.json"),"--device",DEVICE,"--batch-size",str(BATCH_SIZE),"--visual-weight",str(VISUAL_WEIGHT),"--state-weight",str(STATE_WEIGHT)],"retrieval.log")
summary=json.loads((MEMORY_DIR/"heldout_retrieval.json").read_text())["summary"]
print(json.dumps(summary,indent=2))

In [ ]:
# Run all unit tests. Real tests use these paths, including exact action checks.
os.environ["STAGE2_DATA_DIR"]=str(DATA_DIR)
os.environ["STAGE2_MEMORY_DIR"]=str(MEMORY_DIR)
run_logged(["-m","unittest","discover","-s","tests","-v"],"tests.log")
import hashlib
source_files=[*sorted((REPO_DIR/"memory").glob("*.py")),REPO_DIR/"requirements-stage2_2.txt",REPO_DIR/"notebooks/stage2_2_build_memory.ipynb"]
(RUN_DIR/"source_sha256.json").write_text(json.dumps({str(p.relative_to(REPO_DIR)):hashlib.sha256(p.read_bytes()).hexdigest() for p in source_files},indent=2))
# Include the exact HDF5 image-reference targets and split metadata so exports are portable.
# Imported/reproduced data is already under RUN_DIR; otherwise copy it once.
if not DATA_DIR.resolve().is_relative_to(RUN_DIR.resolve()):
    export_data=RUN_DIR/"stage2_1_data"
    export_data.mkdir()
    for name in ("train_demonstrations.hdf5","eval_demonstrations.hdf5","split_manifest.json","dataset_metadata.json"):
        shutil.copy2(DATA_DIR/name,export_data/name)
else: export_data=DATA_DIR
(RUN_DIR/"export_locations.json").write_text(json.dumps({"memory":"memory","stage2_1_data":str(export_data.relative_to(RUN_DIR)),"note":"Resolve image references against stage2_1_data; dataset_directory_hint may refer to original build location"},indent=2))

In [ ]:
archive=Path(shutil.make_archive(str(RUN_DIR),"zip",root_dir=RUN_DIR))
print("Export:",archive,"bytes:",archive.stat().st_size)
persisted=False
if GCS_OUTPUT_URI:
    upload=r"""
from google.cloud import storage
from urllib.parse import urlparse
from pathlib import Path
import sys
u=urlparse(sys.argv[1]);assert u.scheme=='gs' and u.netloc
p=Path(sys.argv[2]);b=storage.Client().bucket(u.netloc).blob(u.path.strip('/')+'/'+p.name)
b.upload_from_filename(str(p),checksum='auto',if_generation_match=0);b.reload()
assert b.size==p.stat().st_size
print('Persisted gs://'+u.netloc+'/'+b.name,'generation',b.generation)
"""
    try:
        subprocess.run([PYTHON,"-c",upload,GCS_OUTPUT_URI,str(archive)],check=True)
        persisted=True
    except subprocess.CalledProcessError:
        print("GCS export failed. Use the explicit download fallback below.")
if not persisted:
    print("Download this archive before stopping the runtime:",archive)
    try:
        from google.colab import files
        files.download(str(archive))
    except (ImportError,RuntimeError):
        from IPython.display import FileLink,display
        display(FileLink(str(archive)))
        print("If the link is inaccessible, download the printed path through the notebook file browser.")

## Completion and next stage
Verify the ZIP exists in your bucket or on your local computer; a download request alone does not prove delivery. Export contains the NPZ, metadata, both modes' full held-out Top-1/Top-5 results, data needed to resolve image references, environment/source provenance and logs. Return the archive or its local extracted location to Codex for inspection.

Similarity and action-distance diagnostics do not establish task success. Before Stage 2.3: verify simulator image conventions and observation adapter, inspect retrieved examples, resolve the policy/expert action comparison space, and design a separately authorized intervention evaluation. Expert success still rests on provenance, not simulator replay.